# afMLevel Demonstration Notebook

This notebook provides an introduction to the **afMLevel** package and demonstrates how to use its trained U‑Net models to level Atomic Force Microscopy (AFM) images. Two machine‑learning–based levelling approaches are demonstrated:

1. **Background Model**
   This model predicts the background artifacts present in an AFM image, such as tilt, line shift and other common imaging artifacts. The predicted
   background is then subtracted from the original image to produce a levelled AFM image. This workflow is implemented in the `level_ml_bg()` function.
   
2. **Mask Model**
   The mask model uses a trained U-Net to generate a feature mask for an AFM image. This mask is then used in a conventional levelling pipeline involving
   plane and line fits, so that the background (unmasked areas) drives the levelling procedure. The mask generation and levelling operations are
   orchestrated by the `level_ml_mask()` function.

To explore how the models work, simply work through this notebook by clicking on each code cell and running it (using **Shift** + **Enter**) to view the output. You can experiment with different parameters by editing the code, and re‑running a cell will immediately show how your changes affect the results. You can also switch from the provided demonstration data to your own AFM datasets by adjusting the file paths. Notebooks are designed for exploration, so you can rerun cells or restart the kernel at any time without worrying about breaking anything.

## 1. Setup and Imports

Start by ensuring all the required packages and functions are installed and imported. The paths to the two trained U-Net models are also set and  `lutAFM` AFM image colourmap loaded.

In [ ]:
# If afMLevel isn't installed ensure this notebook is opened from the
# repository root and uncomment the installation line.
# Install in editable mode if developing:
# !pip install -e .
from pathlib import Path
import os
import requests
from tifffile import imread
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import numpy as np

from afmlevel.background_model import level_ml_bg
from afmlevel.mask_model import level_ml_mask, ml_mask, ml_edges

# Download models if missing and load

def download_if_missing(url: str, dest: Path) -> Path:
    dest.parent.mkdir(parents=True, exist_ok=True)
    if not dest.exists():
        print(f"Downloading model to {dest} ...")
        r = requests.get(url, stream=True)
        r.raise_for_status()
        with open(dest, "wb") as f:
            for chunk in r.iter_content(chunk_size=8192):
                f.write(chunk)
        print("Download complete.")
    else:
        print(f"Model already exists: {dest}")
    return dest

# Links expire 18th April 2026
BG_MODEL_URL = "https://leeds365-my.sharepoint.com/:u:/g/personal/ggjh246_leeds_ac_uk/IQDPSUZk1EdWTpz2lyqE34MyAdF5CIvaMNZl1NBWdh0mGBw?e=hNReZG&download=1"
MASK_MODEL_URL = "https://leeds365-my.sharepoint.com/:u:/g/personal/ggjh246_leeds_ac_uk/IQCy4gUPymjbRLDZWTHo-XghAR1ZY5XkEk-WZL-Une9f5kg?e=0xpco2&download=1"

models_dir = Path("models")

bg_model_path = download_if_missing(BG_MODEL_URL, models_dir / "background_model.pth")
mask_model_path = download_if_missing(MASK_MODEL_URL, models_dir / "mask_model.pth")

# Load AFM colourmap
AFM = np.load('./lutAFM.npy') # by default the working directory is the folder the notebook is in, i.e. /notebooks
AFM = ListedColormap(AFM)

## 2. Loading Data

Next step is to load the AFM image that we will apply the **afMLevel** levelling routines to. We can then visualise the raw data with Matplotlib. 

In [ ]:
# Find repo root so demonstration data can be accessed
def find_repo_root(marker=".git"):
    """Walk up parent directories until we find the repo root (marked by .git folder)."""
    path = Path.cwd()
    for parent in [path] + list(path.parents):
        if (parent / marker).exists():
            return parent
    raise FileNotFoundError("Could not find the repository root (no .git directory found).")

repo_root = find_repo_root()

# Demo path uses sample data from the test suite.
# To use your own data:
#   your_path = Path(r"C:\path\to\your\data.tiff")
#   demo_data = imread(your_path)
demo_path = repo_root / "tests" / "resources" / "sample_0.tiff"
demo_data = imread(demo_path)

plt.figure()
plt.title("Raw AFM Data")
plt.imshow(demo_data, cmap=AFM)
plt.colorbar(label="Height (nm)")
plt.show()


## 4. Apply Models

### Background Model

The background model is applied with the `level_ml_bg()` function within the `background_model` module. The function requires the loaded AFM data (NumPy array) and the path to the trained background model.

If the `background` parameter is set as `True` then rather than returning the levelled image, the background before it is subtracted is returned. 

In [ ]:
# Apply the background model
bg_levelled = level_ml_bg(demo_data, model_path=bg_model_path, line_order=3)

# Apply the background model and return the background
bg_background = level_ml_bg(demo_data, model_path=bg_model_path, line_order=3, background = True)

plt.figure(figsize=(15,4))

plt.subplot(1,3,1)
plt.title("Raw Data")
plt.imshow(demo_data, cmap=AFM)
plt.colorbar(label="Height (nm)")

plt.subplot(1,3,2)
plt.title("Background")
plt.imshow(bg_background, cmap=AFM)
plt.colorbar(label="Height (nm)")

plt.subplot(1,3,3)
plt.title("Levelled Data")
plt.imshow(bg_levelled, cmap=AFM)
plt.colorbar(label="Height (nm)")

plt.tight_layout()
plt.show()

### Mask Model

Masks are generated using the `ml_mask()` and `ml_edges()` functions, which return binary NumPy arrays of type `uint8` (values 0 or 1). The `level_ml_mask()` function coordinates levelling workflows; it applies plane and line‑fit corrections (from the `pnanolocz` library) and uses both afMLevel masking functions to mask features in the data prior to levelling steps. 

#### ml_mask

The `ml_mask()` function detects features within the image and produces a binary mask, enabling those regions to be omitted from the levelling operations.

#### ml_edges

The `ml_edges()` function builds on the output of `ml_mask()` to generate an edge‑specific mask. It first obtains a binary mask from `ml_mask()`, then applies a sequence of morphological operations, including perimeter extraction, removal of small objects, hole filling, and dilation, to isolate the edges of detected features. The resulting binary mask is compatible with the functions in the `level_weighted` module of `pnanolocz`.

In [ ]:
# Generate mask using trained U-Net model

mask = ml_mask(demo_data, model_path=mask_model_path)
edges = ml_edges(demo_data, model_path=mask_model_path)

plt.figure(figsize=(15,4))

plt.subplot(1,3,1)
plt.title("Raw Data")
plt.imshow(demo_data)

plt.subplot(1,3,2)
plt.title("Mask")
plt.imshow(mask)

plt.subplot(1,3,3)
plt.title("Edges Mask")
plt.imshow(edges)

plt.tight_layout()
plt.show()

#### level_ml_mask

These masking functions are used within `level_ml_mask()`. To level an AFM image using the mask model, a levelling routine must be selected from the available methods listed in the `DEFAULT_ML_ROUTINES` dictionary in the `mask_model` module.

In [ ]:
from afmlevel.mask_model import DEFAULT_ML_ROUTINES

routine_names = ", ".join(DEFAULT_ML_ROUTINES)
print(f"Available ML mask levelling routines: {routine_names}.")

In testing, the `iterative Ml mask` routine provided the most consistent results across a wide range of images and is therefore set as the default option.

The levelling routines follow a sequence of levelling and masking steps similar to traditional manual or automated AFM image‑processing workflows. The key difference is that, instead of relying on classical thresholding or edge‑detection methods, the masks are generated by machine‑learning‑based functions (`ml_mask()` and `ml_edges()`), providing more consistent feature detection and more stable levelling performance when applied without manual parameter optimisation for each image.

In [ ]:
method = "iterative ML mask"
steps = DEFAULT_ML_ROUTINES[method]

print(f"Steps for {method}:")
for step in steps:
    name = step["func"].__name__
    if "method" in step:
        name += f" ({step['method']})"
    print(name)

The `level_ml_mask()` takes a NumPy array (`imarray`), the path to the trained mask U-Net model (`model_path`), and a `method` as inputs. 

In [ ]:
# Apply the "iterative ML mask" routine
mask_levelled = level_ml_mask(demo_data, model_path=mask_model_path, method="iterative ML mask")

mask_levelled_mpmil = level_ml_mask(demo_data, model_path=mask_model_path, method="ML mask")


plt.figure(figsize=(15,4))

plt.subplot(1,3,1)
plt.title("Raw Data")
plt.imshow(demo_data, cmap=AFM)
plt.colorbar(label="Height (nm)")

plt.subplot(1,3,2)
plt.title('"iterative ML mask" Levelled')
plt.imshow(mask_levelled, cmap=AFM)
plt.colorbar(label="Height (nm)")

plt.subplot(1,3,3)
plt.title('"ML mask" Levelled')
plt.imshow(mask_levelled_mpmil, cmap=AFM)
plt.colorbar(label="Height (nm)")

plt.tight_layout()
plt.show()